# customer care telegram: practice

This module/project pack is extracted from the local files listed below. Functions and line numbers are source locations; the original program is not executed by practice.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/community_contributions/customer_care_telegram/modules/__init__.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/__init__.py)
- [2_openai/community_contributions/customer_care_telegram/modules/agents/__init__.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/agents/__init__.py)
- [2_openai/community_contributions/customer_care_telegram/modules/agents/order_status_agent.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/agents/order_status_agent.py)
- [2_openai/community_contributions/customer_care_telegram/modules/agents/root_agent.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/agents/root_agent.py)
- [2_openai/community_contributions/customer_care_telegram/modules/agents/sequential_agents.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/agents/sequential_agents.py)
- [2_openai/community_contributions/customer_care_telegram/modules/config.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/config.py)
- [2_openai/community_contributions/customer_care_telegram/modules/in_memory_session.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/in_memory_session.py)
- [2_openai/community_contributions/customer_care_telegram/modules/setup_logging.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/setup_logging.py)
- [2_openai/community_contributions/customer_care_telegram/modules/tools/calculate_order_price.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/tools/calculate_order_price.py)
- [2_openai/community_contributions/customer_care_telegram/modules/tools/check_order_status.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/tools/check_order_status.py)
- [2_openai/community_contributions/customer_care_telegram/modules/tools/run_pandas_query_tool.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/tools/run_pandas_query_tool.py)
- [2_openai/community_contributions/customer_care_telegram/modules/tools/save_order.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/tools/save_order.py)
- [2_openai/community_contributions/customer_care_telegram/modules/tools/send_email.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/tools/send_email.py)
- [2_openai/community_contributions/customer_care_telegram/modules/tools/setup_sheets.py](../../../../../../2_openai/community_contributions/customer_care_telegram/modules/tools/setup_sheets.py)
- [2_openai/community_contributions/customer_care_telegram/README.md](../../../../../../2_openai/community_contributions/customer_care_telegram/README.md)
- [2_openai/community_contributions/customer_care_telegram/requirements.txt](../../../../../../2_openai/community_contributions/customer_care_telegram/requirements.txt)
- [2_openai/community_contributions/customer_care_telegram/telegram_customer_agent.py](../../../../../../2_openai/community_contributions/customer_care_telegram/telegram_customer_agent.py)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 3. An agent loop must make progress and stop

A loop repeats model/tool steps until a final answer or a configured limit. This fixed script of decisions shows the order without executing generated code or calling a model.

**Predict:** Which decision ends the loop?

In [ ]:
decisions = iter([
    {"kind": "tool", "name": "add", "a": 12, "b": 30},
    {"kind": "answer", "text": "The sum is 42."},
])
events = []
for turn in range(3):  # Bound the number of steps.
    decision = next(decisions)
    if decision["kind"] == "tool":
        if decision["name"] != "add":
            raise ValueError("Unknown tool")
        events.append({"tool_result": decision["a"] + decision["b"]})
    else:
        events.append({"final": decision["text"]})
        break
else:
    raise RuntimeError("No final answer within the limit")
print(events)
assert events[0]["tool_result"] == 42

**Check:** The answer decision ends it. A loop that only keeps requesting tools needs a bound and an explicit failure path.

**Change one thing:** Add another tool step before the answer. Keep a finite turn limit and explain how the stopping rule changes.

## 4. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 5. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## 6. Retrieve evidence before answering

A retrieval workflow selects relevant chunks and passes them as context. This tiny keyword example illustrates the boundary without embeddings, a vector store, or a hosted model.

**Predict:** Which chunk matches the query about Python?

In [ ]:
chunks = [{"id": "a", "text": "Alex writes Python automation."},
          {"id": "b", "text": "Alex enjoys hiking."}]
query = "Python"
evidence = [chunk for chunk in chunks if query.lower() in chunk["text"].lower()]
answer_input = {"question": query, "evidence": evidence}
print(answer_input)
assert [chunk["id"] for chunk in evidence] == ["a"]

**Check:** Chunk a. This keyword selection does not implement the source vector/embedding integrations.

**Change one thing:** Try a query with no matches. Return an explicit lack-of-evidence result instead of inventing an answer.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.